## Wi-Fi Access Point Placement at INPT

The objective of this project is to determine the optimal locations for installing Wi-Fi Access Points (APs) inside INPT in Rabat.

The study considers the following areas: Pavilion 1, Pavilion 2, Pavilion 3, the restaurant and classrooms.

The goal is to ensure Wi-Fi coverage in all these areas while minimizing the number of Access Points and their installation cost.

We will use VNS and GVNS to find and compare optimized AP deployment solutions.

In [31]:
import random
from itertools import combinations

## 1. Define the problem

In [32]:
# Installation cost for each candidate location
costs = [10, 15, 12, 8, 20]


coverage = [
    [1, 1, 0, 0],  
    [0, 1, 1, 0], 
    [0, 0, 1, 1],  
    [1, 0, 0, 0],  
    [0, 0, 0, 1]   
]

PENALTY = 100

## 2. Evaluation function

This function calculates the total cost of a solution.

$$
f(S) = \text{Installation Cost}(S) + \text{Uncovered Areas Penalty}(S)
$$

In [33]:
def evaluate(solution):

    total_cost = 0

    for i in range(len(solution)):
        if solution[i] == 1:
            total_cost += costs[i]

    # Check which areas are covered
    uncovered = 0

    for j in range(len(coverage[0])):

        is_covered = False

        for i in range(len(solution)):
            if solution[i] == 1 and coverage[i][j] == 1:
                is_covered = True
                break

        if not is_covered:
            uncovered += 1

    return total_cost + PENALTY * uncovered

## 3. Local Search function
Objective: Improve the solution by exploring its neighbors.

We use the 1-flip neighborhood: change one position from 0 to 1, or from 1 to 0.

In [34]:
def local_search(solution):

    current = solution.copy()

    while True:

        best = current.copy()
        best_cost = evaluate(current)

        # Explore all 1-flip neighbors
        for i in range(len(current)):

            neighbor = current.copy()

            neighbor[i] = 1 - neighbor[i]

            cost = evaluate(neighbor)

            if cost < best_cost:
                best = neighbor
                best_cost = cost

        # Stop if no improvement is possible
        if best == current:
            break

        current = best

    return current

## 4. Shaking function

Objective: Escape local optima by randomly changing the solution.
The parameter k determines how many positions we change.

In [35]:
def shaking(solution, k):

    new_solution = solution.copy()

    positions = random.sample(range(len(solution)), k)

    for i in positions:
        new_solution[i] = 1 - new_solution[i]

    return new_solution

## 5. Main VNS algorithm
Now we combine shaking and local search.

In [36]:
def VNS(initial_solution, kmax, max_iterations):

    best_solution = initial_solution.copy()
    best_cost = evaluate(best_solution)

    for iteration in range(max_iterations):
        k = 1
        while k <= kmax:
            new_solution = shaking(best_solution, k)
            improved_solution = local_search(new_solution)
            new_cost = evaluate(improved_solution)
            
            if new_cost < best_cost:

                best_solution = improved_solution
                best_cost = new_cost

                # Return to the first neighborhood
                k = 1
            else:
                # Move to the next neighborhood
                k += 1
    return best_solution, best_cost

## 6. Execute the algorithm

In [37]:
random.seed(42)

# Initial solution
initial_solution = [1, 0, 0, 0, 0]

# Run VNS
best_solution, best_cost = VNS(
    initial_solution,
    kmax=1,
    max_iterations=1
)

print("Best solution:", best_solution)
print("Minimum cost:", best_cost)

Best solution: [1, 0, 1, 0, 0]
Minimum cost: 22


## GVNS Implementation

The main difference is:
- VNS: Shaking + Local Search.
- GVNS: Shaking + VND.
In GVNS, instead of using a simple **local_search()** function, we use **VND()** to explore multiple neighborhood structures systematically.

In [38]:
def best_neighbor(solution, k):

    best = solution.copy()
    best_cost = evaluate(solution)

    for positions in combinations(range(len(solution)), k):

        neighbor = solution.copy()

        for i in positions:
            neighbor[i] = 1 - neighbor[i]

        cost = evaluate(neighbor)

        if cost < best_cost:
            best = neighbor
            best_cost = cost

    return best

## VND (Local Search)

In [39]:
def VND(solution, neighborhoods):

    current = solution.copy()
    k = 0

    while k < len(neighborhoods):

        neighbor = neighborhoods[k](current)

        if evaluate(neighbor) < evaluate(current):

            current = neighbor
            k = 0

        else:
            k += 1

    return current

## Define VND neighborhoods

In [40]:
neighborhoods = [
    lambda s: best_neighbor(s, 1),  
    lambda s: best_neighbor(s, 2)   
]

In [41]:
def GVNS(initial_solution, kmax, max_iterations):

    best_solution = initial_solution.copy()
    best_cost = evaluate(best_solution)

    for iteration in range(max_iterations):
        k = 1

        while k <= kmax:
            new_solution = shaking(best_solution, k)
            improved_solution = VND(
                new_solution,
                neighborhoods
            )

            new_cost = evaluate(improved_solution)

            if new_cost < best_cost:

                best_solution = improved_solution
                best_cost = new_cost
                k = 1

            else:
                k += 1

    return best_solution, best_cost

## Execute GVNS

In [42]:
random.seed(42)

initial_solution = [1, 0, 0, 0, 0]

gvns_solution, gvns_cost = GVNS(
    initial_solution,
    kmax=3,
    max_iterations=100
)

print("GVNS Best solution:", gvns_solution)
print("GVNS Minimum cost:", gvns_cost)

GVNS Best solution: [1, 0, 1, 0, 0]
GVNS Minimum cost: 22


## Conclusion 
finally we can conclude that the best places to install the access pts 